Comparing Gemini annotations with BiomedParse pseudo-labels
1. Load the existing BiomedParse pseudo-label masks.
2. Identify all connected components without size filtering.
3. Calculate the area, equivalent diameter, maximum and minimum diameters, and bounding-box width and height.
4. Save the measurements to a CSV file.
5. Add Gemini measurements and compare the results across models.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json

drive.mount("/content/drive")

# Directory containing the foundation model experiment results
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"
PREDICTIONS_DIR = PROJECT_DIR / "predictions"

# Load the dataset manifest
manifest = pd.read_csv(MANIFEST_PATH)

# Select the 10 pilot images
pilot = manifest[
    manifest["is_pilot"].astype(str).str.lower() == "true"
].copy()

pilot = pilot.sort_values("case_id").reset_index(drop=True)

# Verify that all pilot annotations are available
assert len(pilot) == 10

for case_id in pilot["case_id"]:
    prediction_path = PREDICTIONS_DIR / f"{case_id}.json"
    assert prediction_path.exists(), f"Missing prediction: {case_id}"

print("Total dataset images:", len(manifest))
print("Pilot images:", len(pilot))
print("Available Gemini annotations:", len(pilot))

display(
    pilot[
        ["case_id", "category", "split", "is_pilot"]
    ]
)

Mount Google Drive and load the dataset manifest for pilot evaluation

In [ ]:
from PIL import Image
import numpy as np

for _, row in pilot.iterrows():

    case_id = row["case_id"]

    image = np.array(Image.open(row["image_path"]))
    mask = np.array(Image.open(row["pseudo_label_path"]))

    # Check spatial dimensions
    assert image.shape[:2] == mask.shape[:2], (
        f"Dimension mismatch: {case_id}"
    )

    unique_values = np.unique(mask)

    print(
        f"{case_id}: "
        f"image={image.shape}, "
        f"mask={mask.shape}, "
        f"labels={unique_values}"
    )

Inspect the BiomedParse pseudo-label masks and verify their dimensions

In [ ]:
from skimage import measure
import numpy as np
from PIL import Image
import pandas as pd

def measure_pseudo_label_regions(mask, case_id):
    """
    Extract all 8-connected foreground components from a binary mask.
    No minimum area filtering is applied.
    """

    binary_mask = mask == 1

    # 8-connectivity for 2D images
    labeled_mask = measure.label(
        binary_mask,
        connectivity=2
    )

    regions = measure.regionprops(labeled_mask)

    records = []

    for region in regions:

        min_row, min_col, max_row, max_col = region.bbox

        records.append({
            "case_id": case_id,
            "model": "BiomedParse",
            "region_id": region.label,
            "area_px2": region.area,
            "equivalent_diameter_px": region.equivalent_diameter_area,
            "major_axis_length_px": region.axis_major_length,
            "minor_axis_length_px": region.axis_minor_length,
            "bbox_width_px": max_col - min_col,
            "bbox_height_px": max_row - min_row,
            "bbox_xmin": min_col,
            "bbox_ymin": min_row,
            "bbox_xmax": max_col,
            "bbox_ymax": max_row
        })

    return records

Extract all connected components and compute geometric measurements from BiomedParse pseudo-labels

In [ ]:
all_measurements = []

for _, row in pilot.iterrows():

    case_id = row["case_id"]

    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    measurements = measure_pseudo_label_regions(
        mask,
        case_id
    )

    all_measurements.extend(measurements)

    print(
        f"{case_id}: {len(measurements)} "
        "connected components"
    )

biomedparse_measurements = pd.DataFrame(all_measurements)

print("\nTotal components:",
      len(biomedparse_measurements))

display(biomedparse_measurements.head(10))

Measure all BiomedParse pseudo-label regions in the pilot dataset

In [ ]:
EVALUATION_DIR = PROJECT_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    EVALUATION_DIR /
    "biomedparse_pilot_measurements.csv"
)

biomedparse_measurements.to_csv(
    output_path,
    index=False
)

print("Measurements saved:", output_path)

Save the BiomedParse region measurements for subsequent model comparison

In [ ]:
from PIL import Image
import pandas as pd
import json

gemini_records = []

for _, row in pilot.iterrows():
    case_id = row["case_id"]

    with open(PREDICTIONS_DIR / f"{case_id}.json", "r", encoding="utf-8") as f:
        result = json.load(f)

    annotation = result["annotation"]

    with Image.open(row["image_path"]) as img:
        width, height = img.size

    for region_id, region in enumerate(annotation["regions"], start=1):
        x1, y1, x2, y2 = region["bbox"]

        # Convert normalized coordinates to original image pixels.
        xmin = x1 / 1000 * width
        ymin = y1 / 1000 * height
        xmax = x2 / 1000 * width
        ymax = y2 / 1000 * height

        bbox_width = xmax - xmin
        bbox_height = ymax - ymin

        gemini_records.append({
            "case_id": case_id,
            "model": result["model"],
            "region_id": region_id,
            "bbox_xmin": xmin,
            "bbox_ymin": ymin,
            "bbox_xmax": xmax,
            "bbox_ymax": ymax,
            "bbox_width_px": bbox_width,
            "bbox_height_px": bbox_height,
            "bbox_area_px2": bbox_width * bbox_height,
            "shape": region["shape"],
            "appearance": region["appearance"]
        })

gemini_measurements = pd.DataFrame(gemini_records)

print("Total Gemini regions:", len(gemini_measurements))
display(gemini_measurements.head(10))

Extract pixel-based bounding-box measurements from the saved Gemini annotations

In [ ]:
gemini_output_path = (
    EVALUATION_DIR / "gemini_pilot_measurements.csv"
)

gemini_measurements.to_csv(
    gemini_output_path,
    index=False
)

print("Gemini measurements saved:", gemini_output_path)

Save Gemini bounding-box measurements for comparison with BiomedParse pseudo-labels

In [ ]:
# Count BiomedParse connected components per image
biomedparse_counts = (
    biomedparse_measurements
    .groupby("case_id")
    .size()
    .rename("biomedparse_count")
)

# Count Gemini bounding boxes per image
gemini_counts = (
    gemini_measurements
    .groupby("case_id")
    .size()
    .rename("gemini_count")
)

# Include all pilot images, including zero-count predictions
count_comparison = (
    pilot[["case_id", "category"]]
    .set_index("case_id")
    .join(biomedparse_counts)
    .join(gemini_counts)
    .fillna({
        "biomedparse_count": 0,
        "gemini_count": 0
    })
    .reset_index()
)

count_comparison["biomedparse_count"] = (
    count_comparison["biomedparse_count"].astype(int)
)

count_comparison["gemini_count"] = (
    count_comparison["gemini_count"].astype(int)
)

# Calculate count differences
count_comparison["absolute_error"] = (
    count_comparison["gemini_count"]
    - count_comparison["biomedparse_count"]
).abs()

count_mae = count_comparison["absolute_error"].mean()

print("Count MAE:", round(count_mae, 3))

display(count_comparison)

Compare follicle-like region counts between Gemini and BiomedParse.
For example, a count MAE of 1.3 means that the number of regions identified by Gemini differs from the number of components identified by BiomedParse by an average of 1.3 regions per image.

In [ ]:
# Combine region-level bounding-box measurements
common_columns = [
    "case_id",
    "model",
    "region_id",
    "bbox_width_px",
    "bbox_height_px"
]

all_regions = pd.concat(
    [
        biomedparse_measurements[common_columns],
        gemini_measurements[common_columns]
    ],
    ignore_index=True
)

# Calculate bounding-box area for both methods
all_regions["bbox_area_px2"] = (
    all_regions["bbox_width_px"]
    * all_regions["bbox_height_px"]
)

# Aggregate measurements per image and model
size_summary = (
    all_regions
    .groupby(["case_id", "model"])
    .agg(
        region_count=("region_id", "count"),
        mean_bbox_width_px=("bbox_width_px", "mean"),
        mean_bbox_height_px=("bbox_height_px", "mean"),
        mean_bbox_area_px2=("bbox_area_px2", "mean"),
        total_bbox_area_px2=("bbox_area_px2", "sum")
    )
    .reset_index()
)

print("Image-level size comparison:")
display(size_summary)

Create an image-level summary of region sizes for both annotation methods

In [ ]:
count_path = EVALUATION_DIR / "pilot_count_comparison.csv"
size_path = EVALUATION_DIR / "pilot_size_comparison.csv"

count_comparison.to_csv(count_path, index=False)
size_summary.to_csv(size_path, index=False)

print("Saved:", count_path)
print("Saved:", size_path)

Save pilot count and bounding-box size comparisons as CSV files

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

IOU_THRESHOLD = 0.5

def calculate_iou(box_a, box_b):
    """Calculate IoU between two bounding boxes in xyxy format."""

    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area_a = max(0, box_a[2] - box_a[0]) * max(
        0, box_a[3] - box_a[1]
    )
    area_b = max(0, box_b[2] - box_b[0]) * max(
        0, box_b[3] - box_b[1]
    )

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def get_boxes(df):
    """Extract bounding boxes from a measurement DataFrame."""

    return df[
        ["bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
    ].to_numpy(dtype=float)


comparison_records = []
matched_records = []

for case_id in pilot["case_id"]:

    bp_regions = biomedparse_measurements[
        biomedparse_measurements["case_id"] == case_id
    ].reset_index(drop=True)

    gem_regions = gemini_measurements[
        gemini_measurements["case_id"] == case_id
    ].reset_index(drop=True)

    bp_boxes = get_boxes(bp_regions)
    gem_boxes = get_boxes(gem_regions)

    n_bp = len(bp_boxes)
    n_gem = len(gem_boxes)

    # Compute pairwise IoU matrix
    iou_matrix = np.zeros((n_bp, n_gem))

    for i, bp_box in enumerate(bp_boxes):
        for j, gem_box in enumerate(gem_boxes):
            iou_matrix[i, j] = calculate_iou(
                bp_box, gem_box
            )

    # One-to-one matching, maximizing the number of
    # threshold-qualified matches before total IoU
    valid_matches = []

    if n_bp > 0 and n_gem > 0:

        matching_score = (
            (iou_matrix >= IOU_THRESHOLD).astype(float) * 2
            + iou_matrix
        )

        row_ind, col_ind = linear_sum_assignment(
            matching_score,
            maximize=True
        )

        for i, j in zip(row_ind, col_ind):
            iou = float(iou_matrix[i, j])

            if iou >= IOU_THRESHOLD:
                valid_matches.append(iou)

                matched_records.append({
                    "case_id": case_id,
                    "biomedparse_region_id": int(
                        bp_regions.iloc[i]["region_id"]
                    ),
                    "gemini_region_id": int(
                        gem_regions.iloc[j]["region_id"]
                    ),
                    "bbox_iou": iou
                })

    tp = len(valid_matches)
    fp = n_gem - tp
    fn = n_bp - tp

    comparison_records.append({
        "case_id": case_id,
        "biomedparse_count": n_bp,
        "gemini_count": n_gem,
        "matched_regions": tp,
        "unmatched_gemini": fp,
        "unmatched_biomedparse": fn,
        "mean_matched_iou": (
            np.mean(valid_matches) if valid_matches else np.nan
        )
    })

spatial_comparison = pd.DataFrame(comparison_records)
matched_regions = pd.DataFrame(matched_records)

display(spatial_comparison)

Calculate bounding-box IoU and match Gemini predictions to BiomedParse pseudo-label regions

In [ ]:
total_tp = int(spatial_comparison["matched_regions"].sum())
total_fp = int(spatial_comparison["unmatched_gemini"].sum())
total_fn = int(spatial_comparison["unmatched_biomedparse"].sum())

precision = (
    total_tp / (total_tp + total_fp)
    if total_tp + total_fp > 0 else 0
)

recall = (
    total_tp / (total_tp + total_fn)
    if total_tp + total_fn > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)

mean_iou = (
    matched_regions["bbox_iou"].mean()
    if not matched_regions.empty else np.nan
)

print("Pilot spatial agreement results")
print("--------------------------------")
print("IoU threshold:", IOU_THRESHOLD)
print("Matched regions:", total_tp)
print("Unmatched Gemini regions:", total_fp)
print("Unmatched BiomedParse regions:", total_fn)
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Mean matched IoU: {mean_iou:.3f}")

Calculate overall spatial agreement metrics for the pilot dataset

In [ ]:

spatial_path = EVALUATION_DIR / "pilot_spatial_comparison.csv"
matches_path = EVALUATION_DIR / "pilot_matched_regions.csv"

spatial_comparison.to_csv(spatial_path, index=False)
matched_regions.to_csv(matches_path, index=False)

print("Saved:", spatial_path)
print("Saved:", matches_path)

Save pilot spatial agreement results for reproducible analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

fig, axes = plt.subplots(5, 2, figsize=(12, 22))
axes = axes.flatten()

for ax, (_, row) in zip(axes, pilot.iterrows()):
    case_id = row["case_id"]

    image = np.array(
        Image.open(row["image_path"]).convert("RGB")
    )

    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    ax.imshow(image)

    # Show BiomedParse pseudo-labels as a blue overlay
    blue_overlay = np.zeros(
        (mask.shape[0], mask.shape[1], 4),
        dtype=float
    )

    # RGB: blue
    blue_overlay[mask == 1, :3] = [0.0, 0.5, 1.0]

    # Transparency only for foreground pixels
    blue_overlay[mask == 1, 3] = 0.45

    ax.imshow(
        blue_overlay,
        interpolation="nearest"
    )

    # Draw Gemini bounding boxes in green
    gem_regions = gemini_measurements[
        gemini_measurements["case_id"] == case_id
    ]

    for _, region in gem_regions.iterrows():
        rect = patches.Rectangle(
            (region["bbox_xmin"], region["bbox_ymin"]),
            region["bbox_width_px"],
            region["bbox_height_px"],
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

    ax.set_title(case_id)
    ax.axis("off")

plt.tight_layout()

# Save the comparison without changing source images
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = FIGURES_DIR / "gemini_vs_biomedparse_pilot.png"

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Figure saved:", figure_path)

Visualize BiomedParse pseudo-labels (blue segmentation mask) and Gemini (green) bounding boxes together

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

fig, axes = plt.subplots(5, 2, figsize=(12, 22))
axes = axes.flatten()

for ax, (_, row) in zip(axes, pilot.iterrows()):
    case_id = row["case_id"]

    image = Image.open(row["image_path"]).convert("RGB")
    ax.imshow(image)

    # BiomedParse bounding boxes
    bp_regions = biomedparse_measurements[
        biomedparse_measurements["case_id"] == case_id
    ]

    for _, region in bp_regions.iterrows():
        rect = patches.Rectangle(
            (region["bbox_xmin"], region["bbox_ymin"]),
            region["bbox_width_px"],
            region["bbox_height_px"],
            linewidth=2,
            edgecolor="dodgerblue",
            facecolor="none"
        )
        ax.add_patch(rect)

    # Gemini bounding boxes
    gem_regions = gemini_measurements[
        gemini_measurements["case_id"] == case_id
    ]

    for _, region in gem_regions.iterrows():
        rect = patches.Rectangle(
            (region["bbox_xmin"], region["bbox_ymin"]),
            region["bbox_width_px"],
            region["bbox_height_px"],
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )
        ax.add_patch(rect)

    ax.set_title(
        f"{case_id} | BP: {len(bp_regions)} | Gemini: {len(gem_regions)}"
    )
    ax.axis("off")

# Add legend
legend_elements = [
    patches.Patch(
        facecolor="none",
        edgecolor="dodgerblue",
        linewidth=2,
        label="BiomedParse pseudo-label bbox"
    ),
    patches.Patch(
        facecolor="none",
        edgecolor="lime",
        linewidth=2,
        label="Gemini predicted bbox"
    )
]

fig.legend(
    handles=legend_elements,
    loc="upper center",
    ncol=2
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

# Save figure to Google Drive
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = FIGURES_DIR / "pilot_bbox_comparison.png"

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Figure saved:", figure_path)

Compare BiomedParse and Gemini bounding boxes on all pilot images

In [ ]:
from scipy.optimize import linear_sum_assignment
from PIL import Image
import numpy as np
import pandas as pd

center_records = []

def box_centers(df):
    x = (df["bbox_xmin"].to_numpy() + df["bbox_xmax"].to_numpy()) / 2
    y = (df["bbox_ymin"].to_numpy() + df["bbox_ymax"].to_numpy()) / 2
    return np.column_stack([x, y])

for _, row in pilot.iterrows():
    case_id = row["case_id"]

    bp = biomedparse_measurements[
        biomedparse_measurements["case_id"] == case_id
    ].reset_index(drop=True)

    gem = gemini_measurements[
        gemini_measurements["case_id"] == case_id
    ].reset_index(drop=True)

    with Image.open(row["image_path"]) as img:
        width, height = img.size

    # Normalize distances by the image diagonal
    diagonal = np.hypot(width, height)

    if bp.empty or gem.empty:
        center_records.append({
            "case_id": case_id,
            "matched_center_pairs": 0,
            "mean_normalized_center_distance": np.nan
        })
        continue

    bp_centers = box_centers(bp)
    gem_centers = box_centers(gem)

    # Pairwise normalized Euclidean distances
    distances = np.linalg.norm(
        bp_centers[:, None, :] - gem_centers[None, :, :],
        axis=2
    ) / diagonal

    # Match nearest centers one-to-one
    bp_indices, gem_indices = linear_sum_assignment(distances)
    matched_distances = distances[bp_indices, gem_indices]

    center_records.append({
        "case_id": case_id,
        "matched_center_pairs": len(matched_distances),
        "mean_normalized_center_distance": float(
            matched_distances.mean()
        )
    })

center_comparison = pd.DataFrame(center_records)

display(center_comparison)

print(
    "Mean normalized center distance:",
    round(
        center_comparison[
            "mean_normalized_center_distance"
        ].mean(),
        4
    )
)

Calculate normalized center distances (average distance between the matched region centers, normalized by the image diagonal) between Gemini and BiomedParse regions

In [ ]:
center_path = (
    EVALUATION_DIR / "pilot_center_distance_comparison.csv"
)

center_comparison.to_csv(center_path, index=False)

print("Saved:", center_path)

Save normalized center-distance measurements for pilot evaluation